# Notebook 06 — Automazione no-code con IA generativa

**Modulo:** 06
**Tempo stimato:** 1.5-2 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Progettato "su carta" (in Markdown) un flusso di automazione completo: trigger, prompt, azione
- Simulato in Python cosa fa realmente uno strumento no-code come Zapier quando esegue un'azione con un LLM
- Costruito un piccolo motore di automazione generico, riutilizzabile su più scenari

> **Nota:** questo modulo non usa Jupyter per costruire l'automazione vera e
> propria — quella si fa in uno strumento come Zapier, fuori da questo
> notebook. Qui, invece, "apriamo il cofano" e vediamo cosa succede quando
> quello strumento chiama un LLM al posto tuo.

## Prerequisiti

```bash
pip install openai python-dotenv
```

Serve un file `.env` nella cartella del progetto con:
```
OPENAI_API_KEY=sk-...
```


---
## Cella 0 — Setup: caricamento della chiave

Esegui sempre questa cella per prima.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK {nome}: {chiave[:8]}...")
    else:
        print(f"MANCANTE {nome}: controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)


---
## Parte 1 — Progettazione testuale di un flusso di automazione

Scenario dato:

> Un e-commerce riceve recensioni dei clienti tramite un modulo web. Ogni
> volta che arriva una recensione con valutazione bassa (1 o 2 stelle su 5),
> vuoi che il sistema generi automaticamente una bozza di risposta empatica
> e professionale, e la invii al team assistenza per la revisione prima
> dell'invio al cliente.

**Prima di scrivere codice**, progetta questo flusso come faresti in Zapier o
n8n. Completa il template qui sotto (modifica direttamente questa cella).

### Il tuo Zap (da completare)

**Trigger:**
- Servizio: ...
- Evento specifico: ...

**Istruzioni di sistema per l'LLM** (le regole permanenti, uguali per ogni
esecuzione):
```
...
```

**Prompt utente** (il contenuto dinamico, proveniente dal trigger — usa
segnaposto tra parentesi graffe per i campi che cambiano ogni volta, es.
`{nome_cliente}`, `{testo_recensione}`, `{valutazione}`):
```
...
```

**Azione finale:**
- Cosa succede all'output generato dall'LLM? (es. "viene inserito come bozza
  in un task di Trello assegnato al team assistenza") ...

**Parametri del modello:**
- `temperature` scelta e perché: ...
- `max_tokens` scelto e perché: ...

---
## Parte 2 — Cosa succede sotto il cofano

Ora simuliamo, in Python, esattamente ciò che uno strumento no-code fa quando
esegue l'azione "invia a un LLM": prende i dati dell'evento (il trigger),
li inserisce in un prompt costruito da un template, chiama l'API, e restituisce
il risultato al passaggio successivo del flusso.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_KEY)

# Un "evento" finto, come se provenisse da un trigger reale (es. un nuovo
# record in un modulo Google Form collegato a Zapier)
evento_recensione = {
    "nome_cliente": "Marco T.",
    "valutazione": 2,
    "testo_recensione": "Il pacco è arrivato con 5 giorni di ritardo rispetto "
                         "alla data promessa, e la scatola era ammaccata. "
                         "Il prodotto dentro per fortuna era integro, ma "
                         "l'esperienza di consegna è stata pessima.",
}

ISTRUZIONI_SISTEMA = (
    "Sei un assistente del servizio clienti di un e-commerce. Scrivi bozze di "
    "risposta a recensioni negative: tono empatico, professionale, senza "
    "promesse che l'azienda non può mantenere. Massimo 4 frasi. Rispondi in "
    "italiano."
)

def costruisci_prompt_utente(evento: dict) -> str:
    """Equivalente del 'template del prompt' che compileresti in Zapier,
    con i campi dinamici del trigger inseriti al posto dei segnaposto."""
    return (
        f"Cliente: {evento['nome_cliente']}\n"
        f"Valutazione: {evento['valutazione']}/5\n"
        f"Recensione: {evento['testo_recensione']}\n\n"
        "Scrivi una bozza di risposta a questa recensione."
    )

risposta = client.chat.completions.create(
    model="gpt-5-mini",
    max_tokens=256,
    temperature=0.5,
    messages=[
        {"role": "system", "content": ISTRUZIONI_SISTEMA},
        {"role": "user", "content": costruisci_prompt_utente(evento_recensione)},
    ]
)

bozza_risposta = risposta.choices[0].message.content
print("Bozza generata:\n")
print(bozza_risposta)
print(f"\nToken usati: {risposta.usage.prompt_tokens} input + {risposta.usage.completion_tokens} output")


Nota come questa cella riproduce esattamente i tre elementi che hai
progettato nella Parte 1: il **trigger** è simulato dal dizionario
`evento_recensione`, il **prompt** è costruito da `costruisci_prompt_utente`,
e l'**azione finale** (nel nostro caso, semplicemente stampare il risultato)
è l'ultima riga. In uno strumento reale, l'azione finale invierebbe questo
testo a un'altra app (Trello, Slack, un'email).

---
## Esercizio — TODO

Adatta il codice della Parte 2 a un secondo scenario: un gestore automatico
di richieste di preventivo. L'evento in arrivo contiene il nome del
richiedente e la descrizione del servizio richiesto; l'LLM deve generare una
breve risposta che conferma la ricezione della richiesta e indica un tempo
di risposta stimato di 48 ore.

In [ ]:
# TODO: definisci un nuovo evento finto per una richiesta di preventivo,
# con almeno i campi "nome_richiedente" e "descrizione_servizio"
evento_preventivo = {
    # ...
}

# TODO: scrivi nuove ISTRUZIONI_SISTEMA_PREVENTIVO adatte a questo scenario

# TODO: scrivi una nuova funzione costruisci_prompt_preventivo(evento) -> str

# TODO: chiama client.chat.completions.create() con i nuovi dati e stampa il risultato


---
## Sfida finale — Un motore di automazione generico

Generalizza quanto fatto finora in una funzione riutilizzabile, che accetta
una lista di eventi e li elabora tutti uno dopo l'altro — proprio come farebbe
uno strumento no-code che processa più esecuzioni dello stesso Zap nel tempo.

Completa i frammenti marcati con `# TODO`.

In [ ]:
def esegui_automazione(
    eventi: list[dict],
    istruzioni_sistema: str,
    costruisci_prompt: callable,
    modello: str = "gpt-5-mini",
) -> list[dict]:
    """
    Simula l'esecuzione ripetuta di un'automazione no-code su una lista di eventi.
    Restituisce una lista di risultati con testo generato e token consumati.
    """
    risultati = []
    costo_totale_token_output = 0

    for evento in eventi:
        prompt_utente = costruisci_prompt(evento)

        risposta = client.chat.completions.create(
            model=modello,
            max_tokens=256,
            temperature=0.5,
            messages=[
                {"role": "system", "content": istruzioni_sistema},
                {"role": "user", "content": prompt_utente},
            ]
        )

        testo = risposta.choices[0].message.content
        # TODO: accumula in costo_totale_token_output i token di output di questa chiamata

        risultati.append({
            "evento": evento,
            "risposta": testo,
            "token_output": risposta.usage.completion_tokens,
        })

    print(f"Eventi processati: {len(eventi)}")
    print(f"Token di output totali: {costo_totale_token_output}")
    return risultati


# --- Prova il motore su una piccola coda di recensioni ---
coda_recensioni = [
    evento_recensione,
    {
        "nome_cliente": "Giulia B.",
        "valutazione": 1,
        "testo_recensione": "Ordine mai arrivato, servizio clienti irraggiungibile per una settimana.",
    },
]

risultati = esegui_automazione(coda_recensioni, ISTRUZIONI_SISTEMA, costruisci_prompt_utente)

for i, r in enumerate(risultati, start=1):
    print(f"\n--- Risultato {i} ({r['evento']['nome_cliente']}) ---")
    print(r["risposta"])


### Estensioni opzionali (se finisci prima)

1. **Filtra per soglia:** modifica `esegui_automazione` in modo che elabori
   solo gli eventi con `valutazione <= 2` (come nello scenario originale),
   ignorando gli altri — replica il comportamento di un filtro Zapier prima
   dell'azione LLM.
2. **Cambia provider:** aggiungi un parametro `provider` alla funzione, per
   poterla eseguire anche su Gemini (approfondiremo questo pattern nel
   Modulo 08 con il wrapper multi-provider).
3. **Simula il ritardo di un piano gratuito:** aggiungi una stampa che
   simuli l'avviso "Esecuzione in coda: fino a 15 minuti di ritardo su piani
   gratuiti", per ricordarti un limite reale degli strumenti no-code.

In [ ]:
# Spazio per le estensioni opzionali


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 06 — Modulo 06 del Corso Completo di IA Generativa*